In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
def dlt_bronze_ingestion(row:Row):

    table_name = row['tableName']
    source_path = row['sourcePath']
    file_format = row['fileFormat']
    schema_location = row['checkpointLocation']

    autoloader_options = {}

    if file_format == "json":
        customer_schema = StructType([
            StructField("customer_id", StringType(), True),
            StructField("first_name", StringType(), True),
            StructField("last_name", StringType(), True),
            StructField("date_of_birth", StringType(), True), # Can cast to DateType later
            StructField("gender", StringType(), True),
            StructField("address", StringType(), True),
            StructField("phone_number", StringType(), True),
            StructField("email", StringType(), True),
            StructField("registration_date", StringType(), True), # Can cast to DateType later
            StructField("preferred_contact_method", StringType(), True)
        ])

        autoloader_options = {
            'cloudFiles.format': 'json',
            'cloudFiles.schemaLocation': schema_location,
            # 'cloudFiles.schemaEvolutionMode': 'addNewColumns', # https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema
            'multiline' : 'true'
            # 'cloudFiles.inferSchema': 'true'
        }

    elif file_format == "csv":
        autoloader_options = {
            'cloudFiles.format': 'csv',
            'cloudFiles.schemaLocation': schema_location,
            'cloudFiles.schemaEvolutionMode': 'addNewColumns', # https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema
            'delimiter' : ',',
            'header': 'true',
            'inferSchema': 'true'
        }

    elif file_format == "parquet":
        autoloader_options = {
            'cloudFiles.format': 'parquet',
            'cloudFiles.schemaLocation': schema_location,
            'cloudFiles.schemaEvolutionMode': 'addNewColumns', # https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema
        }

    else:
        print("Invalid File Format, please pass JSON, CSV or Parquet")

    print(f"Processing source path: {source_path}")
    print(f"File format: {file_format}")
    print(f"Target Bronze table name: {table_name}")
    print(f"Autoloader Schema: {schema_location}")
    print(f"AutoLoader Options: {autoloader_options}")

    if table_name == "products":
        table_properties_dict = {
                "quality": "silver",
                "pipelines.reset.allowed": "false",
                "delta.feature.timestampNtz": "supported"
        }
    else:
        table_properties_dict = {
            "quality": "silver",
            "pipelines.reset.allowed": "false"
        }


    @dlt.table(
        name = table_name,
        comment = f"Raw data for {table_name}",
        table_properties = table_properties_dict
    )
    def bronze_raw_load():

        if file_format == "json":
            df_raw = (
                spark
                .readStream
                .format("cloudFiles")
                .options(**autoloader_options)
                .schema(customer_schema)
                .load(source_path)
                )
        else:
            df_raw = (
                spark
                .readStream
                .format("cloudFiles")
                .options(**autoloader_options)
                .load(source_path)
                )
        
        df = df_raw.withColumn("file_process_time", 
                               date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))

        return df


#### Connecting to Azure SQL DB to read from the control metadata table and parse it to the bronze DLT function

In [0]:
def sql_connection():

    #Server=tcp:azure-sql-db-01.database.windows.net,1433;Initial Catalog=azure-sql-db;Encrypt=True;TrustServerCertificate=False;Connection Timeout=30;Authentication="Active Directory Default";

    #jdbc Url
    jdbc_url = "jdbc:sqlserver://azure-sql-db-01.database.windows.net:1433; database=azure-sql-db"

    connection_properties = {
        "user":"dhivan06",
        "password":"Databricks#123",
        "driver":"com.microsoft.sqlserver.jdbc.SQLServerDriver"
    }

    return connection_properties, jdbc_url

def execute_jdbc_query(query):

    connection_properties, jdbc_url = sql_connection()

    df = (
        spark.read.jdbc(
            url = jdbc_url,
            table = query,
            properties = connection_properties
        )
    )

    return df



#### Query the jdbc table and loop for each table which will be run in parallel DLT pipeline since there is no depedency all these table will dynamically load in bronze layer as delta tables

In [0]:
query = "(SELECT * FROM [dbo].[metadata_control] where groupId = 1) temp" 
    # this can be paramterized also via the pipeline parameter

df = execute_jdbc_query(query)

for row in df.collect():
    dlt_bronze_ingestion(row) 